# 01 CIFAR-10 subset preparation

Prepare the CIFAR-10 subset for a clean-label poisoning experiment.

The setup:
1. Load CIFAR-10 from local folder.
2. Filter to 4 classes: **Plane (0), Car (1), Bird (2), Cat (3)**.
3. Create a smaller subset (500 images per class).
4. Isolate 10 **Target** images (Class T) and remove them from training.
5. Identify 10 **Base** images (Class B) for poisoning.
6. Save the processed data for the next steps.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import os

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries loaded.")

## Load CIFAR-10 data
The notebook reads CIFAR-10 from `data/cifar-10-python`.

In [ ]:
# Define simple transform (convert to tensor)
# We do NOT normalize yet to keep images in [0, 1] range for easy visualization
transform = transforms.Compose([
    transforms.ToTensor()
])

# CIFAR-10 batch files are stored under data/cifar-10-python.
# torchvision expects the folder 'cifar-10-batches-py' inside the root.
# The packaged repo keeps that structure inside data/.
data_root = './data/cifar-10-python'

try:
    trainset_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform)
    testset_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=False, transform=transform)
    print("Successfully loaded CIFAR-10 from local files.")
except RuntimeError:
    print("Could not find data locally. Please ensure the path is correct.")
    # Fallback: download if needed (will verify files exist)
    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset_full = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    print("Loaded/Downloaded CIFAR-10 to ./data")

## Filter and subset data
The experiment keeps four CIFAR-10 classes and takes 500 training images per class.

In [ ]:
# Configuration
CLASSES_TO_KEEP = [0, 1, 2, 3]  # Plane, Car, Bird, Cat
CLASS_NAMES = ['Plane', 'Car', 'Bird', 'Cat']

IMAGES_PER_CLASS = 500

# 0: Plane (Target Class T)
# 2: Bird (Base Class B)
TARGET_CLASS_IDX = 0 
BASE_CLASS_IDX = 2   

def get_filtered_indices(dataset, classes, limit_per_class):
    indices = []
    class_counts = {c: 0 for c in classes}
    
    # Iterate over all data
    for i in range(len(dataset)):
        _, label = dataset[i]
        if label in classes:
            if class_counts[label] < limit_per_class:
                indices.append(i)
                class_counts[label] += 1
                
    return indices

# Get indices for our subset
train_indices = get_filtered_indices(trainset_full, CLASSES_TO_KEEP, IMAGES_PER_CLASS)
test_indices = get_filtered_indices(testset_full, CLASSES_TO_KEEP, 100) # 100 per class for testing

print(f"Subset Training Size: {len(train_indices)} (should be {IMAGES_PER_CLASS * 4})")
print(f"Subset Test Size: {len(test_indices)}")

## Isolate targets and bases
**Targets**: 10 Plane images. We REMOVE these from the training set.
**Bases**: 10 Bird images. We KEEP these in the training set (initially), but note their indices so we can poison them later.

In [ ]:
target_indices_global = [] # Indices in the full CIFAR dataset
base_indices_global = []

NUM_POISONS = 10

# Iterate through our selected subset to find candidates
for idx in train_indices:
    _, label = trainset_full[idx]
    
    # Select Targets (Planes)
    if label == TARGET_CLASS_IDX and len(target_indices_global) < NUM_POISONS:
        target_indices_global.append(idx)
        
    # Select Bases (Birds)
    elif label == BASE_CLASS_IDX and len(base_indices_global) < NUM_POISONS:
        base_indices_global.append(idx)

# IMPORTANT: Remove Targets from the 'train_indices'
# The model must NOT train on the targets.
clean_train_indices = [i for i in train_indices if i not in target_indices_global]

print(f"Selected {len(target_indices_global)} Targets.")
print(f"Selected {len(base_indices_global)} Bases.")
print(f"Final Training Set Size: {len(clean_train_indices)} (Removed targets)")

## Visualize the selected images
Check the selected target and base images before saving the setup.

In [ ]:
def show_images(indices, dataset, title):
    images = [dataset[i][0] for i in indices]
    grid = torchvision.utils.make_grid(images, nrow=5)
    plt.figure(figsize=(10, 4))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(title)
    plt.axis('off')
    plt.show()

print("TARGETS (Class: Plane) - These are held out from training")
show_images(target_indices_global, trainset_full, "Targets (Planes)")

print("BASES (Class: Bird) - These are inside the training set (to be poisoned later)")
show_images(base_indices_global, trainset_full, "Bases (Birds)")

## Save data for the next notebook
The selected indices are saved so the next notebook uses the same setup.

In [ ]:
data_payload = {
    'clean_train_indices': clean_train_indices,
    'test_indices': test_indices,
    'target_indices_global': target_indices_global,
    'base_indices_global': base_indices_global,
    'CLASSES_TO_KEEP': CLASSES_TO_KEEP,
    'class_names': CLASS_NAMES,
    'target_class_idx': TARGET_CLASS_IDX,
    'base_class_idx': BASE_CLASS_IDX
}

torch.save(data_payload, 'step1_data.pt')
print("Data setup saved to 'step1_data.pt'")